# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [4]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')

if api_key and api_key.startswith('AI') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gemini-2.5-flash'
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=api_key)

API key looks good so far


In [5]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/1

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/

In [9]:
def select_relevant_links(url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [10]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'services page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'services page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'project page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'news/blog page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'related product/company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'}]}

In [11]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [12]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 11 relevant links


{'links': [{'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'product page', 'url': 'https://huggingface.co/models'},
  {'type': 'product page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'product page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'documentation page', 'url': 'https://huggingface.co/docs'},
  {'type': 'enterprise solutions page',
   'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'company information page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'learn page', 'url': 'https://huggingface.co/learn'},
  {'type': 'blog page', 'url': 'https://huggingface.co/blog'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [13]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 16 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-Image
Updated
5 days ago
•
7.59k
•
865
zai-org/GLM-4.7-Flash
Updated
about 1 hour ago
•
481
openbmb/AgentCPM-Explore
Updated
1 day ago
•
1.83k
•
364
Lightricks/LTX-2
Updated
about 15 hours ago
•
1.54M
•
1.18k
google/translategemma-4b-it
Updated
5 days ago
•
17.6k
•
354
Browse 2M+ models
Spaces
Running
on
Zero
Featured
979
Qwen Image Multiple Angles 3D Camera
🎥
979
Adjust camera angles in images using 3D controls or sliders
Running
on
Zero
MCP
1.62k
Z Image Turbo
🖼
1.62k
Generate stunning images from text description

In [15]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [17]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 16 relevant links


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nzai-org/GLM-Image\nUpdated\n5 days ago\n•\n7.59k\n•\n865\nzai-org/GLM-4.7-Flash\nUpdated\nabout 1 hour ago\n•\n485\nopenbmb/AgentCPM-Explore\nUpdated\n1 day ago\n•\n1.83k\n•\n364\nLightricks/LTX-2\nUpdated\nabout 15 hours ago\n•\n1.54M\n•\n1.18k\ngoogle/translategemma-4b-it\nUpdated\n5 days ago\n•\n17.6k\n•\n354\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n980\nQwen Image Multip

In [18]:
def create_brochure(company_name, url):
    response = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [19]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 15 relevant links


Hugging Face: The AI Community Building the Future of Machine Learning

Hugging Face is the leading platform where the global machine learning (ML) community converges to create, discover, and collaborate on cutting-edge AI models, datasets, and applications. We are dedicated to democratizing machine learning, making it more accessible, collaborative, and efficient for everyone, from individual researchers to large enterprises.

**What We Offer:**
As the vibrant home of machine learning, Hugging Face provides an unparalleled ecosystem designed to accelerate your AI journey:

*   **Models:** Dive into a vast and diverse repository of over 2 million models, spanning all modalities including text, image, video, audio, and 3D. Explore solutions for tasks such as Text Generation, Image-to-Text, and many more, compatible with leading ML libraries like PyTorch, TensorFlow, and our popular Transformers library.
*   **Datasets:** Access and contribute to a rich collection of over 500,000 datasets, essential for training and refining your machine learning projects.
*   **Spaces (AI Applications):** Discover, run, and share over 1 million interactive AI applications and demos. These "Spaces" provide a powerful way to showcase real-world use cases, prototype rapidly, and engage with the community's latest innovations.
*   **Collaboration Platform:** Host your projects, collaborate seamlessly with peers, and build a strong public ML portfolio by sharing your work with the world.
*   **Open Source Foundation:** Leverage the robust Hugging Face open-source stack to significantly accelerate your development and innovation.
*   **Enterprise Solutions:** For organizations looking to scale their ML operations with confidence, we offer dedicated paid Compute and Enterprise services designed to provide enhanced performance, support, and security.

**Why Hugging Face?**

*   **For Customers:** Gain a significant competitive edge by integrating advanced AI capabilities into your products and services. Benefit from a rich pool of open-source resources, coupled with our enterprise-grade compute and expert support to drive your ML initiatives forward.
*   **For Investors:** Partner with a foundational platform that is actively shaping and driving the open-source AI revolution. Invest in a company fostering a massive, engaged community and providing essential, next-generation tools for the future of machine learning.
*   **For the AI Community & Potential Recruits:** Join a vibrant, collaborative community that champions open science and shared innovation. Contribute to groundbreaking projects, continuously enhance your skills, and build a recognized public profile within the global AI landscape. At Hugging Face, we attract top talent passionate about making a tangible impact on the future of artificial intelligence.

**Our Culture:**
At Hugging Face, our culture is deeply rooted in collaboration, community, and open innovation. We firmly believe that by working together, we can accelerate progress and build a more inclusive, impactful, and accessible future for artificial intelligence. Our platform embodies this ethos, providing the tools and environment for researchers, developers, and AI enthusiasts worldwide to connect, learn, and contribute.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [20]:
def stream_brochure(company_name, url):
    stream = gemini.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [21]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-2.5-flash
Found 8 relevant links


Hugging Face: The AI Community Building the Future

Hugging Face is the leading platform where the global machine learning (ML) community converges to collaborate on innovative models, datasets, and applications. We are dedicated to accelerating the future of AI by providing a robust and collaborative ecosystem for developers, researchers, and organizations worldwide.

**Our Offerings:**

*   **Models:** Explore and contribute to over 2 million ML models across diverse modalities, from language processing to image generation.
*   **Datasets:** Access and share from more than 500,000 datasets, powering the training and refinement of cutting-edge AI.
*   **Spaces:** Discover and deploy over 1 million AI applications, offering interactive demos and tools like image editing, 3D camera adjustments, and text-to-image generation.

**For Individual Innovators & Researchers:**
Hugging Face is your home for machine learning. Create, discover, and collaborate more effectively. You can host unlimited public models, datasets, and applications, move faster with our open-source stack, and explore all modalities including text, image, video, audio, and even 3D. Build your professional ML portfolio by sharing your work with the world.

**For Teams & Enterprises:**
The Hugging Face Team & Enterprise Hub provides the most advanced platform to build AI at scale for your organization. Benefit from enterprise-grade security, comprehensive access controls, and dedicated support. Key features include:

*   **Single Sign-On (SSO):** Securely connect to your identity provider.
*   **Regions:** Select, manage, and audit the location of your repository data.
*   **Audit Logs:** Maintain control with comprehensive logs on all actions taken.

Whether you're starting with a team subscription or exploring flexible enterprise contracts, we provide the infrastructure to accelerate your organization's ML initiatives.

**Our Vision:**
At Hugging Face, we believe in the power of collaboration and open-source innovation. We are fostering an environment where the brightest minds in AI can connect, share, and collectively push the boundaries of what's possible. Join us in building the future of artificial intelligence.

**Company Culture & Careers:**
While specific details on company culture or open job roles are not explicitly detailed on these pages, Hugging Face is fundamentally driven by its mission as "The AI community building the future." This strongly implies a culture rooted in collaboration, innovation, open exchange of knowledge, and a passion for advancing machine learning technology. We invite those eager to contribute to the forefront of AI to explore how they can be a part of our dynamic community.

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>